# Loan Default Scorecard — End-to-End Pipeline
**Cleaning → EDA → Feature Selection → Modelling → Evaluation**

This notebook reproduces the full workflow on the `Loan_Default.csv` dataset (148,670 records, 2019 vintage).
The target is `Status` (1 = default, 0 = non-default), base default rate ≈ 24.6%.

> **Theme of this notebook:** the dataset is riddled with *data leakage*. A large part of the work is
> detecting and removing it. Read the markdown notes — they explain *why* each step is done, not just how.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

# Adjust path as needed
DATA_PATH = 'Loan_Default.csv'

## 1. Load & First Look

We inspect shape, dtypes, the target balance, and missingness before touching anything.

In [ ]:
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
print('\ntarget balance:')
print(df['Status'].value_counts(normalize=True).round(3))
df.head()

In [ ]:
# Missingness — which columns, how much
miss = (df.isna().mean()*100).round(2)
miss[miss > 0].sort_values(ascending=False)

## 2. Leakage Detection #1 — the rate columns

**Why this matters most.** Before any cleaning we check *why* values are missing.
`rate_of_interest`, `Interest_rate_spread`, and `Upfront_charges` are missing almost
exclusively for the defaulted loans — the *absence* of the value encodes the label.

A model using these (or even a "was it missing" flag) would score near-perfectly in
training and collapse in production. **They must be dropped.**

In [ ]:
leak = ['rate_of_interest','Interest_rate_spread','Upfront_charges']
df['rate_missing'] = df['rate_of_interest'].isna()
print(pd.crosstab(df['Status'], df['rate_missing'], normalize='index').round(3))
# -> Status==1 is ~99.5% missing; Status==0 is ~0% missing. Classic leakage.
df = df.drop(columns=leak + ['rate_missing'])
print('dropped:', leak)

## 3. Cleaning

### 3.1 Fix LTV decimal-misplacement outliers
A handful of `LTV` values are 100× too large (e.g. 7831 instead of 78.31). We divide
the >150 values by 100. LTV (loan-to-value %) should realistically sit roughly 1–150.

In [ ]:
df.loc[df['LTV'] > 150, 'LTV'] = df.loc[df['LTV'] > 150, 'LTV'] / 100
print('LTV range now:', round(df['LTV'].min(),2), '-', round(df['LTV'].max(),2))

### 3.2 Impute / drop missing values

- `dtir1`, `LTV`, `income` → **mean imputation** (kept; large share missing).
- `term`, `loan_limit`, and five low-missingness categoricals → **drop rows** (cheap, <3% loss).

> ⚠️ **Caveat we carry forward:** mean-imputing `dtir1` and `LTV` piles many rows onto a
> single value. That artificial spike will later inflate their apparent importance.

In [ ]:
for c in ['dtir1','LTV','income']:
    df[c] = df[c].fillna(df[c].mean())

df = df.dropna(subset=['term','loan_limit'])
df = df.dropna(subset=['approv_in_adv','submission_of_application','age',
                       'loan_purpose','Neg_ammortization'])
print('shape after imputation/drop:', df.shape)

### 3.3 Outlier handling — IQR winsorizing (cap, don't delete)

**Why winsorize instead of delete?** An earlier attempt that *deleted* outliers via a
percentile cut removed ~29% of rows and skewed the default rate from 24.6% to 13.2% —
because defaulted loans disproportionately sit in the tails. Capping (Tukey 1.5×IQR)
keeps every row and preserves the class balance. `Credit_Score` is excluded (it is
effectively uniform — see EDA).

In [ ]:
cont = ['loan_amount','income','property_value','LTV','dtir1']
for c in cont:
    q1, q3 = df[c].quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    df[c] = df[c].clip(lo, hi)
print('rows preserved:', df.shape[0], '| default rate:', round(df['Status'].mean(),4))

## 4. Exploratory Data Analysis

### 4.1 Default rate by category
We use `Credit_Worthiness` (l1/l2) as the grade analog (no literal "grade" column exists).

In [ ]:
for col in ['Credit_Worthiness','loan_purpose']:
    g = df.groupby(col)['Status'].agg(['mean','count'])
    g['mean'] = (g['mean']*100).round(1)
    print(f'\n[{col}]'); print(g)

# term: almost all 360 months; note the anomalous 300-month bucket
tg = df.groupby('term')['Status'].agg(['mean','count'])
tg = tg[tg['count'] >= 100]; tg['mean'] = (tg['mean']*100).round(1)
print('\n[term — default rate %]'); print(tg.sort_values('mean', ascending=False).head())

### 4.2 Distributions of key features by default status

In [ ]:
feats = [('loan_amount','Loan Amount'),('income','Income'),
         ('property_value','Property Value'),('Credit_Score','Credit Score'),
         ('LTV','LTV (%)'),('dtir1','DTI (%)')]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax,(c,lab) in zip(axes.ravel(), feats):
    for s,col,nm in [(0,'#2a9d8f','No default'),(1,'#e76f51','Default')]:
        sns.histplot(df[df['Status']==s][c], bins=40, stat='density', element='step',
                     fill=True, alpha=0.4, color=col, label=nm, ax=ax)
    ax.set_title(lab, fontweight='bold'); ax.set_xlabel(''); ax.legend(fontsize=8)
plt.suptitle('Key Features by Default Status', fontweight='bold'); plt.tight_layout()
plt.show()

**Reading the plots.** `Credit_Score` is flat/identical across classes → no signal
(likely synthetic). The tall spikes at the right edges of loan_amount/income/property_value
are the IQR winsorizing pile-up; the narrow spikes in LTV/DTI are the mean-imputation
artifact. Keep these in mind — they are cleaning footprints, not borrower behaviour.

## 5. Feature Selection — Information Value (IV)

IV quantifies each feature's predictive power for a binary target. Standard reading:
<0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong, **>0.5 suspicious (likely leakage)**.
We drop IV < 0.02.

In [ ]:
def iv_for(s, y, n_bins=10):
    s = s.copy()
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
        s = pd.qcut(s, n_bins, duplicates='drop')
    d = pd.DataFrame({'x': s.astype('object').fillna('NA'), 'y': y})
    g = d.groupby('x')['y'].agg(['sum','count']); g.columns=['bad','tot']; g['good']=g['tot']-g['bad']
    tb, tg = g['bad'].sum(), g['good'].sum()
    g['db'] = g['bad'].replace(0,0.5)/tb; g['dg'] = g['good'].replace(0,0.5)/tg
    return ((g['dg']-g['db'])*np.log(g['dg']/g['db'])).sum()

y = df['Status']
X = df.drop(columns=['Status','ID','year'])
ivs = pd.Series({c: iv_for(X[c], y) for c in X.columns}).sort_values(ascending=False)
print(ivs.round(3))

Note the IV > 0.5 entries: `credit_type` and `property_value`, plus inflated `dtir1`/`LTV`.
These are flags to investigate, **not** wins to celebrate.

## 6. Leakage Detection #2 — `credit_type`

Its IV is astronomically high (>4). We cross-tab it against the target to see why.

In [ ]:
ct = pd.crosstab(df['credit_type'], df['Status'], normalize='index').round(4)
print(ct)
# -> credit_type == 'EQUI' defaults 99.99% of the time. No genuine bureau behaves this way.
# This is a label artifact. Drop it.
drop_iv = ivs[ivs < 0.02].index.tolist()
keep = [c for c in X.columns if c not in drop_iv and c != 'credit_type']
df_model = df[['Status'] + keep].copy()
print('\nfeatures kept:', keep)

## 7. Train / Validation / Test Split (stratified 60/20/20)

**Why not time-based?** All records are from 2019 with no finer date — a true temporal
split is impossible. We use a stratified split (preserves the 24.5% default rate) and fit
**all** preprocessing on train only to avoid leakage across folds.

In [ ]:
yv = df_model['Status']; Xv = df_model.drop(columns=['Status'])
X_tmp, X_test, y_tmp, y_test = train_test_split(Xv, yv, test_size=0.20,
                                                stratify=yv, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25,
                                                  stratify=y_tmp, random_state=RANDOM_STATE)
for nm, yy in [('train',y_train),('val',y_val),('test',y_test)]:
    print(f'{nm:5s} n={len(yy):6d} default_rate={yy.mean():.4f}')

## 8. Modelling — WoE Scorecard (Logistic Regression)

We transform every feature to its Weight of Evidence (WoE) on the training fold, then fit
logistic regression. A `MISSING` bin is created explicitly for `property_value` (the only
feature still carrying NaN). WoE bins + logistic regression = a standard credit scorecard.

In [ ]:
num = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat = [c for c in X_train.columns if c not in num]

def woe_table(s, y):
    d = pd.DataFrame({'x': s.astype('object'), 'y': y})
    g = d.groupby('x', observed=True)['y'].agg(['sum','count']); g.columns=['bad','tot']; g['good']=g['tot']-g['bad']
    tb, tg = g['bad'].sum(), g['good'].sum()
    return np.log((g['good'].replace(0,0.5)/tg)/(g['bad'].replace(0,0.5)/tb)).to_dict()

edges, woe = {}, {}
def num_bin(s, e): return pd.cut(s, e).astype('object').where(s.notna(), 'MISSING')

Xw_tr = pd.DataFrame(index=X_train.index)
for c in num:
    q = np.unique(np.quantile(X_train[c].dropna(), np.linspace(0,1,6))); q[0]=-np.inf; q[-1]=np.inf
    edges[c] = q; b = num_bin(X_train[c], q); woe[c] = woe_table(b, y_train)
    Xw_tr[c] = b.map(woe[c]).astype(float)
for c in cat:
    s = X_train[c].fillna('MISSING'); woe[c] = woe_table(s, y_train)
    Xw_tr[c] = s.map(woe[c]).astype(float)
cols = list(Xw_tr.columns)

def transform(d):
    out = pd.DataFrame(index=d.index)
    for c in num: out[c] = num_bin(d[c], edges[c]).map(woe[c])
    for c in cat: out[c] = d[c].fillna('MISSING').map(woe[c])
    return out.fillna(0.0)[cols]

Xw_val, Xw_test = transform(X_val), transform(X_test)
lr = LogisticRegression(max_iter=1000).fit(Xw_tr, y_train)
print('fitted on', Xw_tr.shape)

## 9. Convert Coefficients to Integer Scorecard Points (PDO scaling)

Standard convention: 600 base points at 50:1 odds, 20 points to double the odds (PDO = 20).
`points = -(coef·WoE + intercept/n)·factor + offset/n`.

In [ ]:
PDO, base, base_odds = 20, 600, 50
factor = PDO/np.log(2); offset = base - factor*np.log(base_odds)
n = len(cols); intercept = lr.intercept_[0]; coef = dict(zip(cols, lr.coef_[0]))

rows = []
for f in cols:
    for binlabel, w in woe[f].items():
        pts = -(coef[f]*w + intercept/n)*factor + offset/n
        rows.append({'feature': f, 'bin': str(binlabel), 'woe': round(w,3), 'points': int(round(pts))})
sc = pd.DataFrame(rows)
spread = sc.groupby('feature')['points'].agg(['min','max'])
spread['range'] = spread['max'] - spread['min']
print(spread.sort_values('range', ascending=False))
# Note: property_value MISSING bin = ~ -194 pts -> the leakage, visible as a huge spread.

## 10. Evaluation — AUC, Gini, KS, Calibration

In [ ]:
def evaluate(Xw, yy, label):
    p = lr.predict_proba(Xw)[:,1]
    auc = roc_auc_score(yy, p); gini = 2*auc - 1
    fpr, tpr, _ = roc_curve(yy, p); ks = np.max(tpr - fpr)
    print(f'[{label}] AUC={auc:.4f}  Gini={gini:.4f}  KS={ks:.4f}')
    return p, fpr, tpr

evaluate(Xw_val, y_val, 'VAL')
p, fpr, tpr = evaluate(Xw_test, y_test, 'TEST')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
# ROC
ax[0].plot(fpr, tpr, color='#e76f51', lw=2, label=f'AUC={roc_auc_score(y_test,p):.3f}')
ax[0].plot([0,1],[0,1],'--', color='gray'); ax[0].set_title('ROC (test)', fontweight='bold')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
# KS
thr = np.linspace(0,1,100)
cg = [np.mean(p[y_test.values==0] <= t) for t in thr]
cb = [np.mean(p[y_test.values==1] <= t) for t in thr]
ax[1].plot(thr, cg, color='#2a9d8f', label='Good CDF'); ax[1].plot(thr, cb, color='#e76f51', label='Bad CDF')
ax[1].set_title('KS curve', fontweight='bold'); ax[1].set_xlabel('Predicted PD'); ax[1].legend()
# Calibration
bins = np.quantile(p, np.linspace(0,1,11)); bins[0]=0; bins[-1]=1
cal = pd.DataFrame({'p':p,'y':y_test.values,'b':np.digitize(p, bins[1:-1])})
cal = cal.groupby('b').agg(pred=('p','mean'), obs=('y','mean'))
ax[2].plot([0,1],[0,1],'--', color='gray', label='Perfect')
ax[2].plot(cal['pred'], cal['obs'], 'o-', color='#264653', label='Model')
ax[2].set_title('Calibration', fontweight='bold'); ax[2].set_xlabel('Predicted'); ax[2].set_ylabel('Observed'); ax[2].legend()
plt.tight_layout(); plt.show()

## 11. Conclusion & Honest Caveat

- Headline test metrics: **AUC ≈ 0.82, Gini ≈ 0.64, KS ≈ 0.50**, with near-perfect calibration.
- **But** the ROC's near-vertical rise and the KS cliff at PD≈1.0 reveal that the
  `property_value` MISSING bin is still leaking label information. The defensible estimate
  of *true* discriminating power is **≈ 0.74 AUC**.
- Credit_Score is inert; dtir1/LTV contribute little once their imputation spike can't be exploited.

**Verdict:** mechanically sound, well-calibrated scorecard — but not deployable until the
MISSING-bin leakage is neutralised and an out-of-time sample is obtained. Its real value is
as a leakage case study.